# Rossmann Store Sales — 01 · ETL Pipeline

I'm working with the Rossmann Store Sales dataset from Kaggle: daily sales for 1,115 drugstores across Germany. The wider project is to forecast a store's daily sales, so what I care about across the whole thing is the `Sales` column over time.

This notebook is only the ETL part. I'm not exploring or modelling anything here, I just want to get the two raw files in, look at what they actually contain, join them, clean up the obvious problems, and save one tidy table that the EDA and forecasting notebooks can load without repeating this work. If a cleaning decision ever changes, I'd rather it live in one place than be copy-pasted across three notebooks.

The order is: download, understand what I have, join, clean, save. I try not to claim anything about the data until I've printed it, so most sections here are "look first, then say what I see".

## 1. Download the data

`kagglehub` pulls the dataset into a local cache and hands back a path, so I don't have to commit the CSVs to the repo. The path is machine-specific, so I use `glob` to find whatever CSVs are in there rather than hardcoding filenames I haven't confirmed exist yet.

In [1]:
import kagglehub
import os, glob
import pandas as pd
import numpy as np

# If this slug ever breaks, search "Rossmann Store Sales" on Kaggle and swap it.
path = kagglehub.dataset_download("pratyushakar/rossmann-store-sales")
print("Path to dataset files:", path)

for p in glob.glob(os.path.join(path, "*.csv")):
    print(" -", os.path.basename(p))

 - train.csv
 - store.csv
 - test.csv


The folder has three CSVs: `train.csv`, `store.csv` and `test.csv`. I'll only load the first two below and see what's in them. `test.csv` is the Kaggle competition's hold-out set with no `Sales` column (that's what competitors were asked to predict), so it's no use for building or forecasting my own series. I'm not assuming that from the name though, it's a known feature of how Kaggle competition datasets are packaged, and if the columns surprise me when I load the files I'll revisit it.

## 2. Understand what's in each file

Before joining or cleaning anything I want to actually look at the two files: how big each one is, what columns it has, and what a single row represents. I'll load both and print shape, columns and the first few rows for each.

In [2]:
train = pd.read_csv(os.path.join(path, "train.csv"), low_memory=False)
store = pd.read_csv(os.path.join(path, "store.csv"))

print("train shape:", train.shape, " (rows, columns)")
print("train columns:", list(train.columns))
print()
train.head()

train shape: (1017209, 9)  (rows, columns)
train columns: ['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday']



,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1


So `train` is about a million rows, and reading the first rows, each one is a single store on a single date with its sales and a few flags. Most column names are self-explanatory, but a couple I can only guess at from the name, so I'll write those as suppositions and check them before relying on them:

| Column | What I read it as |
|---|---|
| `Store` | Store id. I'm assuming this is what links to `store.csv`, to be verified. |
| `DayOfWeek` | Day of week, looks like 1–7. |
| `Date` | The date. |
| `Sales` | Turnover for that store that day. This is my forecasting target. |
| `Customers` | Number of customers that day. |
| `Open` | Looks like 1 = open, 0 = closed. |
| `Promo` | The name suggests a promotion ran that day. I will check the possible values. |
| `StateHoliday` | Some kind of holiday indicator. I don't know its values yet, so I'll print them below before saying what they mean. |
| `SchoolHoliday` | Another holiday indicator. |

A few of these I'm treating as categories or flags (`DayOfWeek`, `Open`, `Promo`, `StateHoliday`, `SchoolHoliday`), so I will print their possible values.

In [14]:
flag_cols = ["DayOfWeek", "Open", "Promo", "StateHoliday", "SchoolHoliday"]
for c in flag_cols:
    vals = sorted(train[c].dropna().unique().tolist(), key=str)
    print(f"{c:14s} unique values: {vals}")

DayOfWeek      unique values: [1, 2, 3, 4, 5, 6, 7]
Open           unique values: [0, 1]
Promo          unique values: [0, 1]
StateHoliday   unique values: ['0', 'a', 'b', 'c']
SchoolHoliday  unique values: [0, 1]


Now I can describe them from what actually printed rather than from a guess:

- `DayOfWeek` runs 1 to 7, so 1 = Monday through 7 = Sunday as expected.
- `Open` is only ever 0 or 1, and `Promo` and `SchoolHoliday` likewise, so those really are clean binary flags, no stray third value to worry about.
- `StateHoliday` is text, not a number: `"0"` for no holiday plus a small set of letter codes (`a`, and on the full dataset also `b` and `c`) for different holiday types. Good to know it's categorical text, because that changes how I'd encode it if I ever use it.


In [4]:
print("store shape:", store.shape, " (rows, columns)")
print("store columns:", list(store.columns))
print()
store.head()

store shape: (1115, 10)  (rows, columns)
store columns: ['Store', 'StoreType', 'Assortment', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval']



,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


`store` is much smaller, 1,115 rows, and the columns look like fixed attributes of a store rather than anything that changes day to day. Again, some names I can read straight off and some I'm guessing at:

| Column | What I read it as |
|---|---|
| `Store` | Store id, presumably the join key. Checking uniqueness below. |
| `StoreType`, `Assortment` | Category codes for the store and its product range. They are letters. I dont know their business meaning. |
| `CompetitionDistance` | Distance to the nearest competitor, going by the name and the magnitude of the numbers (looks like metres). |
| `CompetitionOpenSince[Month/Year]` | When that competitor opened, split into month and year. |
| `Promo2` | A 0/1 flag. The name suggests a second, ongoing kind of promotion, separate from the daily `Promo`, but that's a supposition from the name. |
| `Promo2Since[Week/Year]`, `PromoInterval` | When and how often that Promo2 runs, if the above reading is right. |

Same discipline as before: I called `StoreType` and `Assortment` "category codes". I'll print their actual unique values (and `Promo2`'s), and I'll also check `Store` really is one-per-row here, since a duplicate would multiply sales rows in the join.

In [5]:
# Same as above: confirm the store-file categories rather than trusting .head().
for c in ["StoreType", "Assortment", "Promo2"]:
    print(f"{c:12s} unique values: {sorted(store[c].dropna().unique().tolist(), key=str)}")

# And check Store is one row per store before treating it as the join key.
print()
print("store rows       :", len(store))
print("unique Store ids :", store["Store"].nunique())
print("one row per store:", len(store) == store["Store"].nunique())

StoreType    unique values: ['a', 'b', 'c', 'd']
Assortment   unique values: ['a', 'b', 'c']
Promo2       unique values: [0, 1]

store rows       : 1115
unique Store ids : 1115
one row per store: True


Rows and unique ids match, so `Store` really is one-per-row in the store file, safe to use as the join key (a duplicate here would have quietly multiplied sales rows). And `StoreType` (`a`–`d`), `Assortment` (`a`–`c`) and `Promo2` (0/1) are confirmed as small category sets, so calling them categorical isn't a guess any more. I still don't know what each letter *means* in business terms, only that they're categories, which is all I need for ETL.

So the data is split the way transactional data usually is: the events over time (`train`) in one file, the fixed attributes of the thing (`store`) in another. Putting them together is the next step.

## 3. Join the two tables

Every daily row has a `Store`, but the store's attributes live in the other file, so I attach them with a left join on `Store`.

I'm using a left join on purpose rather than an inner one. A left join keeps every `train` row no matter what, so I can't silently lose sales history, which is the thing I most want to avoid in ETL. The trade-off is that a left join won't *tell* me if some rows failed to match, it'll just leave the store columns null for them. So the honest check isn't the row count (a left join can't change that by construction, so it proves nothing on its own), it's counting how many rows came out with no store attributes attached. If that's zero, every row found a match and I got the safety of `left` and the guarantee of `inner` at once.

In [6]:
df = train.merge(store, on="Store", how="left")

print("train rows :", len(train))
print("merged rows:", len(df))
print("columns after join:", df.shape[1])

# The real check: did any train row fail to find a matching store?
# StoreType comes only from store.csv, so a null there means "no match".
unmatched = df["StoreType"].isnull().sum()
print("rows with no matching store (null StoreType):", unmatched)

train rows : 1017209
merged rows: 1017209
columns after join: 18
rows with no matching store (null StoreType): 0


Zero unmatched rows, so every daily record found its store and nothing was dropped or left with blank attributes. That's the thing the row count alone couldn't tell me. The table is now one row per store per day with the store attributes attached.

## 4. Parse dates and sort

`Date` comes in as text. Later notebooks need it as a real datetime, to pull out the weekday, to sort in time order, and to slice by date, so I convert it now. I also sort by store and then date, so each store's history is in chronological order. That ordering matters for anything that looks backwards in time, like the lag features I build for forecasting in notebook 03 (a lag is "sales 7 days ago", which is only correct if the rows are actually in date order).

In [7]:
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Store", "Date"]).reset_index(drop=True)

print("date range :", df["Date"].min().date(), "->", df["Date"].max().date())
print("days covered:", df["Date"].nunique())
df[["Store", "Date", "Sales", "Customers", "Open", "Promo"]].head()

date range : 2013-01-01 -> 2015-07-31
days covered: 942


,Store,Date,Sales,Customers,Open,Promo
0,1,2013-01-01,0,0,0,0
1,1,2013-01-02,5530,668,1,0
2,1,2013-01-03,4327,578,1,0
3,1,2013-01-04,4486,619,1,0
4,1,2013-01-05,4997,635,1,0


The printed range covers early 2013 to mid-2015, roughly two and a half years. I'm not going to claim anything about seasonality here since I haven't plotted anything yet (that's the whole point of notebook 02), but it's worth noting the span is long enough that a full year repeats more than once, which is the minimum I'd want before trying to forecast a yearly pattern later. For now that's just a note about coverage, not a finding.

## 5. Clean

Cleaning here is about making the table trustworthy, not just making code run. Looking at what I've seen so far, there are three things I want to deal with, and I'll take them one at a time:

1. **Closed days** — rows where the store wasn't open. I need to know what their sales look like and how many there are.
2. **Missing values** — several store-attribute columns had nulls when I glanced at them. I want to see exactly which columns, and decide what a null means in each before filling anything.
3. **A plausibility pass on `Sales`** — check there's nothing impossible, like negative turnover, before I trust the target.

Taking them in that order.

### 5a. Closed days

When a store is closed, I'd expect its sales to be zero. If that's true, these are structural zeros, the store wasn't trading so there was nothing to sell, which is different from a store that opened and happened to sell nothing. I want to confirm sales really are zero on closed days, and see how many such rows there are. I'll keep them in the saved table, but the EDA and forecasting notebooks will filter to open days, because averaging closed days in would drag every figure down and invent a fake weekly dip.

In [8]:
closed_mask = df["Open"] == 0
print(f"closed-day rows: {closed_mask.sum():,} ({closed_mask.mean():.1%} of all rows)")
print("max sales on closed days:", df.loc[closed_mask, "Sales"].max())

# I want to say something about *which* days these are, so break the closed rows down by weekday.
print("\nclosed rows by DayOfWeek (1=Mon ... 7=Sun):")
print(df.loc[closed_mask, "DayOfWeek"].value_counts().sort_index())

closed-day rows: 172,817 (17.0% of all rows)
max sales on closed days: 0

closed rows by DayOfWeek (1=Mon ... 7=Sun):
DayOfWeek
1      7170
2      1703
3      3729
4     11201
5      7205
6       672
7    141137
Name: count, dtype: int64


The max sales on closed days is 0, so closed really does mean zero turnover, these are structural zeros, not missing values recorded as 0. And the weekday breakdown makes one thing obvious: the closed rows are overwhelmingly `DayOfWeek == 7` (Sunday), which fits the fact that shops in Germany are essentially all shut on Sundays.

That still leaves a chunk of closed days on the *other* weekdays. My first guess is public holidays, so let me actually check it against the `StateHoliday` column below rather than assume.

In [15]:
# Are the non-Sunday closures explained by public holidays?
# Sundays have their own reason (national closure), so exclude them and look at the rest.
non_sun = df[df["DayOfWeek"] != 7].copy()
non_sun["is_closed"]  = non_sun["Open"] == 0
non_sun["is_holiday"] = non_sun["StateHoliday"] != "0"   # "0" = no holiday

print("Non-Sunday rows — closed vs state holiday:")
print(pd.crosstab(non_sun["is_closed"], non_sun["is_holiday"]))

closed_ns = non_sun[non_sun["is_closed"]]
print(f"\nOf non-Sunday CLOSED rows, share that fall on a state holiday: "
      f"{closed_ns['is_holiday'].mean():.1%}")

# The closed, non-holiday leftovers: what weekdays are they on?
leftover = closed_ns[~closed_ns["is_holiday"]]
print(f"Closed, non-Sunday, non-holiday rows: {len(leftover):,}")
print("their spread across weekdays (1=Mon ... 6=Sat):")
print(leftover["DayOfWeek"].value_counts().sort_index())

Non-Sunday rows — closed vs state holiday:
is_holiday   False  True 
is_closed                
False       839891    908
True          1847  29833

Of non-Sunday CLOSED rows, share that fall on a state holiday: 94.2%
Closed, non-Sunday, non-holiday rows: 1,847
their spread across weekdays (1=Mon ... 6=Sat):
DayOfWeek
1    369
2    300
3    287
4    344
5    268
6    279
Name: count, dtype: int64


Reading the table, my first guess mostly holds, and now I've got the numbers behind it instead of a hunch:

* State holidays close stores. Almost every non-Sunday state-holiday row is closed, so when a holiday falls on a weekday, the store is shut.
* And the reverse holds too: the non-Sunday closures are overwhelmingly holidays. About 94% of the weekday closed rows fall on a state holiday, so once Sundays are set aside, "closed on a weekday" almost always means "public holiday".
* That leaves only a small remainder (1,847 rows) closed on ordinary, non-holiday weekdays, and those are spread fairly evenly across Monday to Saturday. An even spread like that isn't holidays (holidays would clump on whichever weekdays specific dates land on); it's more consistent with the odd individual store shut for a stretch, a refurbishment or temporary closure, which touches every weekday roughly equally as the weeks pass.

So the honest summary is: Sundays are the bulk of closures (national closure), and essentially all the remaining weekday closures are public holidays, with a small tail of per-store temporary closures. That confirms the original guess. None of it changes what I do next (these are all genuine closed days and I filter them out downstream), but I'd rather have checked it than assumed it.

### 5b. Missing values

When I looked at `store` earlier a few columns had nulls. Rather than fill anything blindly I want to print exactly which columns have nulls and how many, then decide per column what the null actually means.

In [10]:
nulls = df.isnull().sum()
print("columns with nulls:")
print(nulls[nulls > 0])

columns with nulls:
CompetitionDistance            2642
CompetitionOpenSinceMonth    323348
CompetitionOpenSinceYear     323348
Promo2SinceWeek              508031
Promo2SinceYear              508031
PromoInterval                508031
dtype: int64


Every null sits in a store-attribute column, none in `Sales` itself, which is the reassuring part since `Sales` is what I'm forecasting.

Reading what the nulls mean: a missing `CompetitionDistance` looks like "no competitor recorded", and the missing `Promo2Since...` / `PromoInterval` values line up with stores that aren't in Promo2 (there's simply no schedule to record). So these are "not applicable" rather than "we failed to measure it", and filling them with 0 (or `"None"` for the text column) is honest, it's not inventing a measurement the way filling a missing *sales* figure with 0 would.

Worth being straight about the alternative: for the forecasting I'm planning I probably won't even use most of these competition/Promo2 columns, so I could just leave them as-is. I'm filling them anyway so the saved table has no nulls at all and any later notebook can use them without tripping over missing values, but if I were sure I'd never touch them, leaving them null would be defensible too.

In [11]:
fill_zero = ["CompetitionDistance", "CompetitionOpenSinceMonth", "CompetitionOpenSinceYear",
             "Promo2SinceWeek", "Promo2SinceYear"]
for c in fill_zero:
    if c in df.columns:
        df[c] = df[c].fillna(0)

# PromoInterval is text, so its "not applicable" marker is a word, not 0.
if "PromoInterval" in df.columns:
    df["PromoInterval"] = df["PromoInterval"].fillna("None")

print("remaining nulls:", int(df.isnull().sum().sum()))

remaining nulls: 0


No nulls left. Every value in the table is now either a real measurement or an explicit "not applicable".

### 5c. Plausibility check on Sales

Last thing before saving: a quick sanity pass on the target itself. Sales should never be negative, and the range on open days should look like believable daily store turnover rather than something obviously broken. I'm checking open days specifically because closed days are all zero and would just drag the minimum to 0 for a reason I already understand.

In [12]:
open_sales = df.loc[df["Open"] == 1, "Sales"]
print("negative sales rows:", (df["Sales"] < 0).sum())
print(f"sales on open days -> min: {open_sales.min():,}  "
      f"median: {open_sales.median():,.0f}  max: {open_sales.max():,}")

# I don't want to call zero-sales open days "rare" without counting them.
zero_open = (open_sales == 0).sum()
print(f"open days with zero sales: {zero_open:,} ({zero_open / len(open_sales):.3%} of open days)")

negative sales rows: 0
sales on open days -> min: 0  median: 6,369  max: 41,551
open days with zero sales: 54 (0.006% of open days)


No negative sales, and the median and max look like plausible daily turnover for a drugstore. The count above is the honest version of what I'd have otherwise just asserted: open days with exactly zero sales are a tiny fraction of open days (the printed percentage), probably refurbishment days or data gaps. Because it's such a small share, I'm not treating it as a problem here, it won't move the aggregate series I forecast, but I've counted it rather than hand-waved it, and if it turns out to matter in the EDA I can come back to this.

## 6. Save the clean table

I'm saving to Parquet instead of CSV. I've used CSV before and it works, so this is a deliberate choice rather than a habit: CSV throws away dtypes, so `Date` would come back as text every single time and every downstream notebook would have to re-parse it (and re-fix any other types). Parquet stores the schema alongside the data, so `Date` stays a datetime when I reload it, and the file is smaller and faster to read on top of that. The one thing I give up is human-readability, I can't open a Parquet file in a text editor the way I can a CSV, but for an intermediate file that only my notebooks read, that doesn't cost me anything.

In [13]:
os.makedirs("data", exist_ok=True)
out_path = "data/rossmann_clean.parquet"
df.to_parquet(out_path, index=False)

print("saved:", out_path)
print("shape:", df.shape)
print(f"size on disk: {os.path.getsize(out_path) / 1e6:.1f} MB")

saved: data/rossmann_clean.parquet
shape: (1017209, 18)
size on disk: 4.0 MB


## Summary

What this notebook did, and the checks that back each step:

- Loaded `train` and `store`, and before treating `Store` as the key I confirmed it's one row per store in the store file.
- Printed the actual unique values of every flag/category column (`Open`, `Promo`, `StateHoliday`, `SchoolHoliday`, `StoreType`, `Assortment`, `Promo2`), so nothing categorical is described from a guess. I suppose `Promo`s meaning is a flag to indicate whetehr there was a promo that day on that store or not.
- Joined with a left join and then proved every row matched by counting nulls in a store-only column, which the row count alone couldn't show.
- Parsed dates and sorted chronologically, which the lag features in notebook 03 depend on.
- Confirmed closed days are structural zeros (sales are 0 when `Open == 0`) and checked by weekday that they're overwhelmingly Sundays, kept them in the table, and left the filtering to the later notebooks.
- Filled the store-attribute nulls with explicit "not applicable" markers, while noting I may not use those columns at all.
- Ran a plausibility check on `Sales`: no negatives, believable range, and counted the zero-sales open days rather than assuming they were rare.
- Saved `data/rossmann_clean.parquet` as the single clean table the rest of the project reads.

Next up is `02_exploratory_analysis`, where I actually look at the series, weekly and yearly patterns, and whether that `Promo` flag has any impact on sales, before deciding how to model the sales forecast with ML